In [4]:

# ============================================================
# PRACTICAL 4: DATA WRANGLING
# DATA WRANGLING ON REAL ESTATE MARKET
# Dataset: Bengaluru_House_Data.csv
# ============================================================


# 1. IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


# 2. IMPORT THE DATASET

df1 = pd.read_csv("Bengaluru_House_Data.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df1.shape)

display(df1.head())


# 3. CLEAN COLUMN NAMES

df1.columns = (
    df1.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned Column Names:")
print(df1.columns.tolist())


# 4. EXPLORE THE DATASET

print("\nDataset Information:")
df1.info()

print("\nStatistical Summary:")
display(df1.describe(include="all").T)

print("\nUnique Area Types:")
print(df1["area_type"].unique())

print("\nArea Type Counts:")
print(df1["area_type"].value_counts())


# 5. HANDLE MISSING VALUES

# Remove extra spaces from text columns

for col in df1.select_dtypes(include="object").columns:
    df1[col] = df1[col].str.strip()

print("\nMissing Values Before Treatment:")
print(df1.isnull().sum())

# Remove records with missing location
df1 = df1.dropna(subset=["location"])

# Fill missing society values
df1["society"] = df1["society"].fillna("Unknown")

# Fill missing size values with mode
df1["size"] = df1["size"].fillna(df1["size"].mode()[0])

# Fill missing numerical values with median
df1["bath"] = df1["bath"].fillna(df1["bath"].median())
df1["balcony"] = df1["balcony"].fillna(df1["balcony"].median())

# Fill missing availability and area type
df1["availability"] = df1["availability"].fillna("Unknown")
df1["area_type"] = df1["area_type"].fillna(
    df1["area_type"].mode()[0]
)

print("\nMissing Values After Treatment:")
print(df1.isnull().sum())


# 6. REMOVE DUPLICATE RECORDS

print("\nDuplicate Records Before Removal:")
print(df1.duplicated().sum())

df1 = df1.drop_duplicates()

print("\nDuplicate Records After Removal:")
print(df1.duplicated().sum())


# 7. CREATE A CLEANED DATAFRAME

# Keep important columns for analysis

df2 = df1.drop(
    ["society", "balcony", "availability"],
    axis="columns"
)

print("\nUpdated Dataset Shape:")
print(df2.shape)

display(df2.head())


# 8. EXTRACT BHK FROM SIZE COLUMN

# Convert values like 2 BHK and 4 Bedroom into numbers

df2["bhk"] = df2["size"].apply(
    lambda x: int(re.findall(r"\d+", str(x))[0])
)

print("\nUnique BHK Values:")
print(df2["bhk"].unique())

print("\nProperties with more than 20 BHK:")
display(df2[df2["bhk"] > 20])


# 9. CONVERT TOTAL_SQFT INTO NUMERIC VALUES

# Handle ranges and measurement units

def convert_sqft_to_num(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower().replace(",", "")

    # Conversion factors into square feet
    units = {
        "sq. meter": 10.7639,
        "sq meter": 10.7639,
        "sq meters": 10.7639,
        "sq. meters": 10.7639,
        "sq. yard": 9,
        "sq yard": 9,
        "sq yards": 9,
        "sq. yards": 9,
        "perch": 272.25,
        "acres": 43560,
        "acre": 43560,
        "grounds": 2400,
        "ground": 2400,
        "cents": 435.6,
        "cent": 435.6,
        "guntha": 1089
    }

    for unit, factor in units.items():

        if unit in value:

            numbers = re.findall(
                r"\d+(?:\.\d+)?", value
            )

            if numbers:
                return float(numbers[0]) * factor

    # Extract numerical values
    numbers = re.findall(
        r"\d+(?:\.\d+)?", value
    )

    # Handle ranges such as 2100-2850
    if len(numbers) >= 2:
        return (
            float(numbers[0]) + float(numbers[1])
        ) / 2

    elif len(numbers) == 1:
        return float(numbers[0])

    return np.nan


df2["total_sqft"] = df2["total_sqft"].apply(
    convert_sqft_to_num
)

# Remove rows where square feet could not be converted

df2 = df2.dropna(subset=["total_sqft"])

print("\nConverted Total Square Feet:")
display(df2.head())


# 10. CREATE PRICE PER SQUARE FOOT

df2["price_per_sqft"] = (
    df2["price"] * 100000 / df2["total_sqft"]
)

print("\nPrice Per Square Foot Statistics:")
display(df2["price_per_sqft"].describe())


# 11. STANDARDIZE LOCATION NAMES

df2["location"] = df2["location"].apply(
    lambda x: x.strip()
)

location_stats = df2["location"].value_counts(
    ascending=False
)

print("\nTotal Unique Locations:")
print(len(location_stats))

# Find locations having 10 or fewer records

location_stats_less_than_10 = location_stats[
    location_stats <= 10
]

print("\nLocations Having 10 or Fewer Records:")
print(len(location_stats_less_than_10))

# Group less frequent locations into other

df2["location"] = df2["location"].apply(
    lambda x: "other"
    if x in location_stats_less_than_10
    else x
)

print("\nUnique Locations After Standardization:")
print(df2["location"].nunique())


# 12. FILTER AND SUBSET THE DATA

# Remove properties having less than 300 sqft per BHK

df3 = df2[
    ~(df2["total_sqft"] / df2["bhk"] < 300)
].copy()

print("\nShape After Filtering:")
print(df3.shape)

# Filter properties with prices between
# the 10th and 90th percentile

lower_price = df3["price"].quantile(0.10)
upper_price = df3["price"].quantile(0.90)

filtered_data = df3[
    (df3["price"] >= lower_price) &
    (df3["price"] <= upper_price)
]

print("\nFiltered Property Records:")
display(filtered_data.head())


# 13. DATA AGGREGATION

# Average price by location

location_stats = (
    df3.groupby("location")["price"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Price By Location:")
display(location_stats.head(15))

# Average price by area type

area_stats = (
    df3.groupby("area_type")["price"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Price By Area Type:")
display(area_stats)

# Average price by BHK

bhk_stats = (
    df3.groupby("bhk")["price"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Price By BHK:")
display(bhk_stats)


# 14. DATA MERGING (OPTIONAL)

# If an additional neighborhood dataset is available,
# merge it using the common location column.

import os

if os.path.exists("./datasets/Neighborhood_Demographics.csv"):

    extra_df = pd.read_csv(
        "./datasets/Neighborhood_Demographics.csv"
    )

    extra_df.columns = (
        extra_df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    if "location" in extra_df.columns:

        extra_df["location"] = (
            extra_df["location"].astype(str).str.strip()
        )

        df3 = pd.merge(
            df3,
            extra_df,
            on="location",
            how="left"
        )

        print("\nAdditional Dataset Merged Successfully!")

    else:
        print("\nLocation column not found in additional dataset.")

else:

    print("\nNo Additional Dataset Available.")
    print("Merging Step Skipped.")


# 15. IDENTIFY AND HANDLE OUTLIERS USING IQR

# First visualize outliers

outlier_columns = [
    "total_sqft",
    "bath",
    "price",
    "bhk",
    "price_per_sqft"
]

for col in outlier_columns:

    plt.figure(figsize=(7, 4))

    plt.boxplot(df3[col].dropna())

    plt.title("Boxplot of " + col)
    plt.ylabel(col)

    plt.show()


# Remove outliers using IQR method

df4 = df3.copy()

for col in outlier_columns:

    Q1 = df4[col].quantile(0.25)
    Q3 = df4[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Find outliers
    outliers = df4[
        (df4[col] < lower_limit) |
        (df4[col] > upper_limit)
    ]

    print(
        "\nOutliers in", col,
        ":", len(outliers)
    )

    # Remove outlier records
    df4 = df4[
        (df4[col] >= lower_limit) &
        (df4[col] <= upper_limit)
    ].copy()

print("\nDataset Shape After Outlier Removal:")
print(df4.shape)


# 16. VISUALIZE CLEANED DATA

# House price distribution

plt.figure(figsize=(8, 5))

plt.hist(df4["price"], bins=30)

plt.title("Distribution of House Prices")
plt.xlabel("Price in Lakhs")
plt.ylabel("Frequency")

plt.show()


# Average price by area type

plt.figure(figsize=(8, 5))

sns.barplot(
    data=df4,
    x="area_type",
    y="price",
    estimator="mean"
)

plt.title("Average House Price by Area Type")
plt.xlabel("Area Type")
plt.ylabel("Average Price")

plt.xticks(rotation=20)
plt.show()


# Area versus price

plt.figure(figsize=(8, 5))

plt.scatter(
    df4["total_sqft"],
    df4["price"]
)

plt.title("Total Square Feet vs House Price")
plt.xlabel("Total Square Feet")
plt.ylabel("Price in Lakhs")

plt.show()


# 17. CATEGORICAL ENCODING

# One-hot encoding for categorical columns

categorical_cols = [
    "location",
    "area_type"
]

encoded_df = pd.get_dummies(
    df4,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("\nEncoded Dataset:")
display(encoded_df.head())

print("\nEncoded Dataset Shape:")
print(encoded_df.shape)


# 18. TRAIN-TEST SPLIT

# Prepare input features and target variable

# Do not use price_per_sqft as an input feature
# because it is calculated using the target price.

X = df4[
    [
        "location",
        "area_type",
        "total_sqft",
        "bath",
        "bhk"
    ]
]

y = df4["price"]

# Encode categorical input features

X = pd.get_dummies(
    X,
    columns=["location", "area_type"],
    drop_first=True,
    dtype=int
)

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=10
)

print("\nTraining Data Shape:")
print(X_train.shape)

print("\nTesting Data Shape:")
print(X_test.shape)

print("\nTraining Target Shape:")
print(y_train.shape)

print("\nTesting Target Shape:")
print(y_test.shape)


# 19. EXPORT THE WRANGLED DATASET

df4.to_csv(
    "./datasets/RealEstate_Wrangled_Data.csv",
    index=False
)

encoded_df.to_csv(
    "./datasets/RealEstate_Encoded_Data.csv",
    index=False
)

filtered_data.to_csv(
    "./datasets/RealEstate_Filtered_Data.csv",
    index=False
)

location_stats.to_csv(
    "./datasets/Average_Price_By_Location.csv"
)

print("\nAll Files Exported Successfully!")


# 20. FINAL VERIFICATION

print("\nFinal Dataset Shape:")
print(df4.shape)

print("\nMissing Values:")
print(df4.isnull().sum().sum())

print("\nDuplicate Records:")
print(df4.duplicated().sum())

print("\nFinal Cleaned Dataset:")
display(df4.head())




FileNotFoundError: [Errno 2] No such file or directory: 'Bengaluru_House_Data.csv'